# GOR-v3 Round 2 Research Study: Dual-Bank + GOR-v2 & Selective AttnRes + GOR-v2

Nghiên cứu tiếp nối từ kết quả NVIDIA L4 Round 1, kiểm chứng 2 kiến trúc lai mới cùng các mốc đối chứng chuẩn trên cả hai mặt trận:
1. **Stage 1 — Deep Residual Matrix Benchmark (`L = 24, d = 128`):** So sánh `pre_norm`, `gor_v2` (tối ưu kernel không tạo tensor trung gian), `std_dual_bank`, `dual_bank_gor_v2` (Dual-Bank + unbounded GOR-v2 $\phi \le 2$), `attn_res_block`, và `attn_res_gor_v2` (Block AttnRes + GOR-v2 chọn lọc ở nhánh Attention).
2. **Stage 2 — Real From-Scratch Causal Transformer Pretraining (`L = 16`, `WikiText-103` + `TinyStories` OOD):** Đo Val Loss, Val PPL, Val BPB, OOD BPB, Norm Growth, Depth Renewal (`1 - CKA`), và Throughput (`tok/s`).

In [ ]:
import os
import torch

os.system("nvidia-smi")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

os.system("pip install -q datasets transformers matplotlib")

In [ ]:
import os
import subprocess

repo_dir = "/content/Alba-model"
if os.path.exists(repo_dir):
    subprocess.run(["rm", "-rf", repo_dir], check=True)
subprocess.run(
    ["git", "clone", "-b", "Khang", "https://github.com/potatips/Alba-model.git", repo_dir],
    check=True,
)
%cd /content/Alba-model
os.system("python3 -m unittest gor_research.test_gor_research -v")

In [ ]:
%cd /content/Alba-model
!python3 -m gor_research.run_colab_study --device cuda --variants pre_norm,gor_v2,std_dual_bank,dual_bank_gor_v2,attn_res_block,attn_res_gor_v2 --output-dir /content/gor_from_scratch_results

In [ ]:
import json
from IPython.display import Image, display

res_dir = "/content/gor_from_scratch_results"
with open(f"{res_dir}/gor_from_scratch_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("META:", json.dumps(data["meta"]))
print("\n=== STAGE 1: MATRIX STRUCTURED L=24 ===")
for k, v in data["stage1_matrix"]["structured_l24"].items():
    print(f"{k:16s} | mse={v['best_val_mse']:.5f} | cos={v['best_val_cos']:.4f} | oh={v['overhead_params']}")

print("\n=== STAGE 2: FROM-SCRATCH CAUSAL LM ===")
for k, v in data["stage2_lm"].items():
    print(
        f"{k:16s} | val_loss={v['final_val_loss']:.4f} | ppl={v['final_val_ppl']:.2f} | "
        f"bpb={v['final_val_bpb']:.4f} | ood_bpb={v['ood_bpb']:.4f} | "
        f"norm={v['norm_growth']:.2f}x | renew={v['avg_depth_renewal']:.4f} | "
        f"tok_s={v['tok_per_sec']:.0f}"
    )

display(Image(filename=f"{res_dir}/gor_from_scratch_report.png"))

In [ ]:
from google.colab import files

res_dir = "/content/gor_from_scratch_results"
files.download(f"{res_dir}/gor_from_scratch_results.json")
files.download(f"{res_dir}/gor_from_scratch_report.png")
files.download(f"{res_dir}/gor_from_scratch_report.html")